In [47]:
import pandas as pd
import plotly.express as px
import folium
import yaml
import numpy as np

import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=False)

In [48]:
model = calliope.read_yaml('model.yaml')

[2026-03-18 18:57:57] INFO     Math init | loading pre-defined math.
[2026-03-18 18:57:57] INFO     Math init | loading math files {'storage_inter_cluster', 'milp', 'operate', 'base', 'spores'}.
[2026-03-18 18:57:57] INFO     Model: preprocessing data
[2026-03-18 18:57:57] INFO     Math build | building applied math with ['base', 'operate'].
[2026-03-18 18:57:57] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2026-03-18 18:57:57] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2026-03-18 18:57:58] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2026-03-18 18:57:58] INFO     input data `link_from` not defined in model math; it will not be available in the optimisation problem.
[2026-03-18 18:57:58] INFO     input data `color` not defined in model math; it will not be available in the optimisation probl

In [49]:
model.inputs

<xarray.Dataset> Size: 81kB
Dimensions:                     (costs: 1, techs: 8, nodes: 5, carriers: 1,
                                 timesteps: 168)
Coordinates:
  * costs                       (costs) object 8B 'monetary'
  * techs                       (techs) object 64B 'X0_to_X1' ... 'supply_gri...
  * carriers                    (carriers) object 8B 'electricity'
  * nodes                       (nodes) object 40B 'X0' 'X1' 'X2' 'X3' 'X4'
  * timesteps                   (timesteps) datetime64[ns] 1kB 2024-04-01 ......
Data variables: (12/35)
    cost_interest_rate          (costs) float64 8B 0.1
    bigM                        float64 8B 1e+06
    objective_cost_weights      (costs) float64 8B 1.0
    base_tech                   (techs) object 64B 'transmission' ... 'supply'
    carrier_in                  (nodes, techs, carriers) bool 40B True ... False
    color                       (techs) object 64B '#823739' ... '#C5ABE3'
    ...                          ...
    cost_flow_out               (costs, timesteps, techs) float64 11kB nan .....
    sink_use_equals             (timesteps, techs, nodes) float64 54kB nan .....
    definition_matrix           (nodes, techs, carriers) bool 40B True ... False
    distance                    (techs) float64 64B 0.6028 0.5168 ... nan nan
    timestep_resolution         (timesteps) float64 1kB 1.0 1.0 1.0 ... 1.0 1.0
    timestep_weights            (timesteps) float64 1kB 1.0 1.0 1.0 ... 1.0 1.0

In [50]:
model.inputs.flow_cap_max.to_series().dropna()  

techs
X0_to_X1             100000.0
X0_to_X2             100000.0
X0_to_X3             100000.0
X0_to_X4             100000.0
battery               50000.0
pv                     1000.0
supply_grid_power     50000.0
Name: flow_cap_max, dtype: float64

In [51]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

techs               nodes
demand_electricity  X1       1.116057e+05
                    X2       8.400000e+05
                    X3       1.344000e+06
                    X4       2.016000e+06
Name: sink_use_equals, dtype: float64

In [52]:
model.build(force=True)

[2026-03-18 18:57:58] INFO     Model: backend build starting
[2026-03-18 18:57:58] INFO     Optimisation Model | parameters/lookups | Generated.
[2026-03-18 18:57:58] INFO     Optimisation Model | variables | Generated.
[2026-03-18 18:57:59] INFO     Optimisation Model | global_expressions | Generated.
[2026-03-18 18:58:00] INFO     Optimisation Model | constraints | Generated.
[2026-03-18 18:58:00] INFO     Optimisation Model | piecewise_constraints | Generated.
[2026-03-18 18:58:00] INFO     Optimisation Model | objectives | Generated.
[2026-03-18 18:58:00] INFO     Model: backend build complete


In [53]:
model.backend.parameters

<xarray.Dataset> Size: 13kB
Dimensions:                             (costs: 1, techs: 8, timesteps: 24,
                                         nodes: 5)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 64B 'X0_to_X1' ... 'su...
  * timesteps                           (timesteps) datetime64[ns] 192B 2024-...
  * nodes                               (nodes) object 40B 'X0' 'X1' ... 'X4'
Data variables: (12/65)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 64B parameters[...
    ...                                  ...
    area_use                            float64 8B nan
    storage_cap                         (techs) object 64B nan nan ... nan nan
    purchased_units                     float64 8B nan
    source_cap                          float64 8B nan
    flow_cap                            (techs) object 64B nan ... parameters...
    link_flow_cap                       float64 8B nan

In [54]:
model.solve(solver='gurobi')

[2026-03-18 18:58:00] INFO     Optimisation model | starting model in operate mode.
[2026-03-18 18:58:00] INFO     Optimisation model | Running first time window.
[2026-03-18 18:58:00] INFO     Optimisation model | Running time window starting at 2024-04-01 12:00:00.
[2026-03-18 18:58:00] INFO     Optimisation model | parameters:storage_initial | The optimisation problem components ['balance_storage', 'set_storage_initial'] will be re-built.
[2026-03-18 18:58:01] INFO     Optimisation model | Running time window starting at 2024-04-02 00:00:00.
[2026-03-18 18:58:01] INFO     Optimisation model | Running time window starting at 2024-04-02 12:00:00.
[2026-03-18 18:58:01] INFO     Optimisation model | Running time window starting at 2024-04-03 00:00:00.
[2026-03-18 18:58:01] INFO     Optimisation model | Running time window starting at 2024-04-03 12:00:00.
[2026-03-18 18:58:01] INFO     Optimisation model | Running time window starting at 2024-04-04 00:00:00.
[2026-03-18 18:58:01] INFO   

In [55]:
model.backend.parameters

<xarray.Dataset> Size: 7kB
Dimensions:                             (costs: 1, techs: 8, timesteps: 12,
                                         nodes: 5)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 64B 'X0_to_X1' ... 'su...
  * timesteps                           (timesteps) datetime64[ns] 96B 2024-0...
  * nodes                               (nodes) object 40B 'X0' 'X1' ... 'X4'
Data variables: (12/65)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 64B parameters[...
    ...                                  ...
    storage_cap                         (techs) object 64B nan nan ... nan nan
    purchased_units                     float64 8B nan
    source_cap                          float64 8B nan
    flow_cap                            (techs) object 64B nan ... parameters...
    link_flow_cap                       float64 8B nan
    storage_initial                     (nodes, techs) object 320B nan ... nan

In [56]:
model.results

<xarray.Dataset> Size: 553kB
Dimensions:                     (nodes: 5, techs: 8, carriers: 1,
                                 timesteps: 168, costs: 1)
Coordinates:
  * techs                       (techs) object 64B 'X0_to_X1' ... 'supply_gri...
  * timesteps                   (timesteps) datetime64[ns] 1kB 2024-04-01 ......
  * nodes                       (nodes) object 40B 'X0' 'X1' 'X2' 'X3' 'X4'
  * carriers                    (carriers) object 8B 'electricity'
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/15)
    flow_out                    (nodes, techs, carriers, timesteps) float64 54kB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 54kB ...
    flow_export                 (nodes, techs, carriers, timesteps) float64 54kB ...
    source_use                  (nodes, techs, timesteps) float64 54kB nan .....
    storage                     (nodes, techs, timesteps) float64 54kB nan .....
    flow_out_inc_eff            (nodes, techs, carriers, timesteps) float64 54kB ...
    ...                          ...
    min_cost_optimisation       (timesteps) float64 1kB 2.494e+06 ... 4.335e+05
    capacity_factor             (nodes, techs, carriers, timesteps) float64 54kB ...
    systemwide_capacity_factor  (techs, carriers) float64 64B 0.0 0.0 ... 0.9297
    systemwide_levelised_cost   (timesteps, techs, costs, carriers) float64 11kB ...
    total_levelised_cost        (timesteps, costs, carriers) float64 1kB 0.57...
    unmet_sum                   float64 8B nan

In [57]:
costs = model.results.cost.to_series().dropna()
costs.head()

timesteps            nodes  techs              costs   
2024-04-01 00:00:00  X0     supply_grid_power  monetary    2.493523e+06
2024-04-01 01:00:00  X0     supply_grid_power  monetary    2.493523e+06
2024-04-01 02:00:00  X0     supply_grid_power  monetary    2.493523e+06
2024-04-01 03:00:00  X0     supply_grid_power  monetary    2.493523e+06
2024-04-01 04:00:00  X0     supply_grid_power  monetary    2.493523e+06
Name: cost, dtype: float64

In [58]:
lcoes = (
    model.results.systemwide_levelised_cost.sel(carriers="electricity")
    .to_series()
    .dropna()
)
lcoes.head()

timesteps            techs              costs   
2024-04-01 00:00:00  supply_grid_power  monetary    0.570197
2024-04-01 01:00:00  supply_grid_power  monetary    0.570197
2024-04-01 02:00:00  supply_grid_power  monetary    0.570197
2024-04-01 03:00:00  supply_grid_power  monetary    0.570197
2024-04-01 04:00:00  supply_grid_power  monetary    0.570197
Name: systemwide_levelised_cost, dtype: float64

In [59]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

In [60]:
df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

      techs           timesteps  Flow in/out (kWh)
0  X0_to_X1 2024-04-01 00:00:00          -3.318958
1  X0_to_X1 2024-04-01 01:00:00          -3.350278
2  X0_to_X1 2024-04-01 02:00:00          -3.363862
3  X0_to_X1 2024-04-01 03:00:00          -3.363862
4  X0_to_X1 2024-04-01 04:00:00          -3.363359


In [61]:
carriers = ["electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

  nodes     techs     carriers           timesteps  Flow in/out (kWh)
0    X0  X0_to_X1  electricity 2024-04-01 00:00:00        -549.509252
1    X0  X0_to_X1  electricity 2024-04-01 01:00:00        -554.694834
2    X0  X0_to_X1  electricity 2024-04-01 02:00:00        -556.944002
3    X0  X0_to_X1  electricity 2024-04-01 03:00:00        -556.944002
4    X0  X0_to_X1  electricity 2024-04-01 04:00:00        -556.860701


In [62]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

AttributeError: 'Dataset' object has no attribute 'flow_cap'

In [ ]:
with open("model.yaml", "r", encoding="utf-8") as f:
    model_def = yaml.safe_load(f)

node_techs = {
    node: list(node_data.get("techs", {}).keys())
    for node, node_data in model_def.get("nodes", {}).items()
}

In [ ]:
# Build a simple system map (nodes + links)
nodes = pd.read_csv("nodes_coordinates.csv")
links = pd.read_csv("links_techs.csv")

flow_cap = (
    model.results.flow_cap.to_series().dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)
flow_cap_lookup = dict(zip(flow_cap["techs"], flow_cap["Flow capacity (kW)"]))

center = [nodes.latitude.mean(), nodes.longitude.mean()]
system_map = folium.Map(location=center, zoom_start=15, tiles="CartoDB dark_matter")

# Add link lines
for _, row in links.iterrows():
    from_row = nodes.loc[nodes.nodes == row["link_from"]].iloc[0]
    to_row = nodes.loc[nodes.nodes == row["link_to"]].iloc[0]
    capacity = flow_cap_lookup.get(row["techs"], row.get("flow_cap_max"))
    popup = (
        f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}"
    )
    if capacity is not None:
        popup += f"<br>Capacity: {capacity:.2f} kW"
    folium.PolyLine(
        locations=[[from_row.latitude, from_row.longitude], [to_row.latitude, to_row.longitude]],
        color=row.get("color", "#1f77b4"),
        weight=3,
        opacity=1,
        popup=popup,
    ).add_to(system_map)

# Add node markers
color_map = model.inputs.color.to_series().to_dict()
for _, row in nodes.iterrows():
    node = row["nodes"]
    techs = node_techs.get(node, [])
    base_types = (
        model.inputs.base_tech.sel(techs=techs).to_series().to_dict()
        if techs
        else {}
    )

    node_type = "Other"
    if any(t == "demand" for t in base_types.values()):
        node_type = "Demand"
    elif any(t == "supply" for t in base_types.values()):
        node_type = "Supply"

    marker_color = "#666666"
    for tech in techs:
        if tech in color_map:
            marker_color = color_map[tech]
            break

    popup = f"<b>{node}</b> ({node_type})<br>Techs: {', '.join(techs)}"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=marker_color,
        fill=True,
        fillColor=marker_color,
        fillOpacity=1,
        popup=popup,
    ).add_to(system_map)

system_map.save("system_map.html")

In [ ]:
output_folder = os.path.join(os.getcwd(), "outputs")
os.makedirs(output_folder, exist_ok=True)
output_csv = os.path.join(output_folder, "bill_of_materials.csv")

# 1) Flow capacities per node/tech (kW)
cap_s = (
    model.results.flow_cap.to_series()
    .dropna()
    .where(lambda x: x != 0)
    .dropna()
)
cap_df = cap_s.to_frame("capacity_kw").reset_index()

# 2) Collapse carriers (if present) so we get exactly one row per (node, tech)
if {"nodes", "techs"}.issubset(cap_df.columns):
    group_cols = ["nodes", "techs"]
else:
    raise ValueError(f"Unexpected flow_cap index columns: {cap_df.columns.tolist()}")

if "carriers" in cap_df.columns:
    cap_node_tech = (
        cap_df.groupby(group_cols, as_index=False, dropna=False)
        .agg(
            capacity_kw=("capacity_kw", "sum"),
            carriers=("carriers", lambda s: ",".join(sorted({str(x) for x in s.dropna()}))),
        )
    )
else:
    cap_node_tech = (
        cap_df.groupby(group_cols, as_index=False, dropna=False)
        .agg(capacity_kw=("capacity_kw", "sum"))
    )

# 3) Tech metadata (unique by tech)
tech_meta = model.inputs.base_tech.to_series().rename("base_tech").reset_index()

if "name" in model.inputs:
    tech_meta = tech_meta.merge(
        model.inputs.name.to_series().rename("name").reset_index(),
        how="left",
        on="techs",
    )
else:
    tech_meta["name"] = pd.NA

if "distance" in model.inputs:
    tech_meta = tech_meta.merge(
        model.inputs.distance.to_series().rename("distance_km").reset_index(),
        how="left",
        on="techs",
    )
else:
    tech_meta["distance_km"] = pd.NA

tech_meta["distance_m"] = tech_meta["distance_km"] * 1000.0
tech_meta["name"] = tech_meta["name"].fillna(tech_meta["techs"])

# 4) Join + filter to links + supply + storage
df = cap_node_tech.merge(
    tech_meta[["techs", "name", "base_tech", "distance_m"]],
    how="left",
    on="techs",
)

allowed_base_tech = {"transmission", "supply", "storage"}
df = df[df["base_tech"].fillna("").str.lower().isin(allowed_base_tech)].copy()

# 5) Export (keeps nodes + techs, no cross-node aggregation)
sort_cols = [c for c in ["nodes", "base_tech", "name", "techs"] if c in df.columns]
df = df.sort_values(by=sort_cols).reset_index(drop=True)

df.to_csv(output_csv, index=False)

NameError: name 'os' is not defined